In [2]:
from typing import Optional, List, Tuple


class AVLNode:
    """AVL树节点类"""
    def __init__(self, val: int):
        self.val = val
        self.left: Optional[AVLNode] = None
        self.right: Optional[AVLNode] = None
        self.height: int = 1  # 节点高度（叶子节点高度为1）


class AVLTree:
    """AVL树"""
    def __init__(self):
        self.root: Optional[AVLNode] = None
        self.steps: List[str] = []  # 记录操作步骤

    def get_height(self, node: Optional[AVLNode]) -> int:
        """获取节点高度"""
        return node.height if node else 0

    def get_balance(self, node: Optional[AVLNode]) -> int:
        """获取节点的平衡因子（左子树高度 - 右子树高度）"""
        if not node:
            return 0
        return self.get_height(node.left) - self.get_height(node.right)

    def update_height(self, node: AVLNode) -> None:
        """更新节点高度"""
        node.height = 1 + max(self.get_height(node.left), self.get_height(node.right))

    def right_rotate(self, y: AVLNode) -> AVLNode:
        """
        RR旋转（右旋）
        适用于：左子树过高（LL型失衡）
        """
        x = y.left
        T2 = x.right

        # 执行旋转
        x.right = y
        y.left = T2

        # 更新高度
        self.update_height(y)
        self.update_height(x)

        return x

    def left_rotate(self, x: AVLNode) -> AVLNode:
        """
        LL旋转（左旋）
        适用于：右子树过高（RR型失衡）
        """
        y = x.right
        T2 = y.left

        # 执行旋转
        y.left = x
        x.right = T2

        # 更新高度
        self.update_height(x)
        self.update_height(y)

        return y

    def insert(self, val: int, verbose: bool = True) -> None:
        """插入节点并保持AVL平衡"""
        if verbose:
            print(f"\n{'='*60}")
            print(f"插入节点: {val}")
            print(f"{'='*60}")

        self.root = self._insert(self.root, val, verbose)

        if verbose:
            self.print_tree_with_balance()
            print()

    def _insert(self, node: Optional[AVLNode], val: int, verbose: bool = True) -> AVLNode:
        """递归插入内部方法"""
        # 1. 标准BST插入
        if not node:
            if verbose:
                print(f"  -> 创建新节点 {val}")
            return AVLNode(val)

        if val < node.val:
            if verbose:
                print(f"  -> {val} < {node.val}，插入左子树")
            node.left = self._insert(node.left, val, verbose)
        elif val > node.val:
            if verbose:
                print(f"  -> {val} > {node.val}，插入右子树")
            node.right = self._insert(node.right, val, verbose)
        else:
            # 重复值不插入
            if verbose:
                print(f"  -> {val} 已存在，不插入")
            return node

        # 2. 更新当前节点的高度
        self.update_height(node)

        # 3. 获取平衡因子
        balance = self.get_balance(node)

        if verbose:
            print(f"  -> 节点 {node.val}: 高度={node.height}, 平衡因子={balance}")

        # 4. 判断失衡类型并旋转

        # LL型：左子树过高，且插入在左子树的左侧
        if balance > 1 and val < node.left.val:
            if verbose:
                print(f"  ⚠️ 节点 {node.val} 发生 LL 型失衡！")
                print(f"  -> 执行右旋 (RR旋转)")
                print(f"  -> 旋转轴: {node.left.val}")
            return self.right_rotate(node)

        # RR型：右子树过高，且插入在右子树的右侧
        if balance < -1 and val > node.right.val:
            if verbose:
                print(f"  ⚠️ 节点 {node.val} 发生 RR 型失衡！")
                print(f"  -> 执行左旋 (LL旋转)")
                print(f"  -> 旋转轴: {node.right.val}")
            return self.left_rotate(node)

        # LR型：左子树过高，但插入在左子树的右侧
        if balance > 1 and val > node.left.val:
            if verbose:
                print(f"  ⚠️ 节点 {node.val} 发生 LR 型失衡！")
                print(f"  -> 先对左子树 {node.left.val} 执行左旋")
                print(f"  -> 再对节点 {node.val} 执行右旋")
                print(f"  -> 旋转轴: {node.left.right.val if node.left.right else '?'}")
            node.left = self.left_rotate(node.left)
            return self.right_rotate(node)

        # RL型：右子树过高，但插入在右子树的左侧
        if balance < -1 and val < node.right.val:
            if verbose:
                print(f"  ⚠️ 节点 {node.val} 发生 RL 型失衡！")
                print(f"  -> 先对右子树 {node.right.val} 执行右旋")
                print(f"  -> 再对节点 {node.val} 执行左旋")
                print(f"  -> 旋转轴: {node.right.left.val if node.right.left else '?'}")
            node.right = self.right_rotate(node.right)
            return self.left_rotate(node)

        return node

    def inorder_traversal(self, node: Optional[AVLNode] = None) -> List[int]:
        """中序遍历（返回有序序列）"""
        if node is None:
            node = self.root

        result = []

        def inorder(n: Optional[AVLNode]):
            if n:
                inorder(n.left)
                result.append(n.val)
                inorder(n.right)

        inorder(node)
        return result

    def print_tree_with_balance(self, node: Optional[AVLNode] = None, prefix: str = "", is_left: bool = True, is_root: bool = True):
        """打印树的形态，并在节点旁标注平衡因子"""
        if node is None:
            node = self.root

        if node is None:
            print("空树")
            return

        balance = self.get_balance(node)

        if is_root:
            print(f"{node.val}(bf={balance})")
            if node.left or node.right:
                if node.left:
                    self.print_tree_with_balance(node.left, "", True, False)
                else:
                    print("├── None")
                if node.right:
                    self.print_tree_with_balance(node.right, "", False, False)
                else:
                    print("└── None")
        else:
            print(prefix + ("├── " if is_left else "└── ") + f"{node.val}(bf={balance})")
            child_prefix = prefix + ("│   " if is_left else "    ")
            if node.left or node.right:
                if node.left:
                    self.print_tree_with_balance(node.left, child_prefix, True, False)
                else:
                    print(child_prefix + "├── None")
                if node.right:
                    self.print_tree_with_balance(node.right, child_prefix, False, False)
                else:
                    print(child_prefix + "└── None")

    def get_ascii_with_balance(self) -> str:
        """生成带平衡因子的ASCII树形图"""
        lines = []

        def build_lines(node: Optional[AVLNode], prefix: str = "", is_left: bool = True):
            if node is None:
                return
            balance = self.get_balance(node)
            lines.append(prefix + ("├── " if is_left else "└── ") + f"{node.val}(bf={balance})")
            child_prefix = prefix + ("│   " if is_left else "    ")
            if node.left:
                build_lines(node.left, child_prefix, True)
            if node.right:
                build_lines(node.right, child_prefix, False)

        if self.root:
            balance = self.get_balance(self.root)
            lines.append(f"{self.root.val}(bf={balance})")
            if self.root.left:
                build_lines(self.root.left, "", True)
            if self.root.right:
                build_lines(self.root.right, "", False)
        return "\n".join(lines)

    def print_step_summary(self, step_num: int, val: int):
        """打印每一步的总结"""
        print(f"\n{'─'*40}")
        print(f"第 {step_num} 步完成")
        print(f"当前树的中序遍历: {self.inorder_traversal()}")
        print(f"{'─'*40}")


# ==================== 主程序 ====================
if __name__ == "__main__":
    print("=" * 70)
    print("AVL 树构建与旋转实操")
    print("插入序列: [30, 20, 10, 25, 40, 35, 50]")
    print("=" * 70)

    avl = AVLTree()
    insert_sequence = [30, 20, 10, 25, 40, 35, 50]

    for i, val in enumerate(insert_sequence, 1):
        avl.insert(val, verbose=True)
        avl.print_step_summary(i, val)

    # 最终结果
    print("\n" + "=" * 70)
    print("最终 AVL 树")
    print("=" * 70)
    print("\n树形结构（带平衡因子）：")
    print(avl.get_ascii_with_balance())

    print("\n树的层级结构图：")
    print("""
          30(bf=0)
         /        \\
       20(bf=0)   40(bf=0)
       /   \\      /   \\
     10(0) 25(0) 35(0) 50(0)
    """)

    inorder = avl.inorder_traversal()
    print(f"\n中序遍历结果: {inorder}")
    print(f"验证BST性质: {'✓ 通过' if inorder == sorted(inorder) else '✗ 失败'}")
    print("(中序遍历得到递增序列，BST性质保持)")

    # 详细步骤说明
    print("\n" + "=" * 70)
    print("详细步骤总结")
    print("=" * 70)
    print("""
┌─────────────────────────────────────────────────────────────────────┐
│ 步骤1: 插入 30                                                        │
│   -> 树: 30(bf=0)                                                     │
│   -> 平衡: 无失衡                                                     │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤2: 插入 20                                                        │
│   -> 树:   30(bf=1)                                                   │
│         /                                                           │
│       20(bf=0)                                                       │
│   -> 平衡: 无失衡                                                     │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤3: 插入 10                                                        │
│   -> 插入前: 30(bf=1) -> 插入10后: 30(bf=2) 失衡！                      │
│   -> 失衡类型: LL型（左左）                                            │
│   -> 旋转轴: 20                                                        │
│   -> 旋转: 右旋（RR旋转）                                              │
│   -> 结果:     20(bf=0)                                               │
│             /    \\                                                  │
│           10(0)  30(0)                                               │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤4: 插入 25                                                        │
│   -> 树:     20(bf=0)                                                 │
│           /    \\                                                    │
│         10(0)  30(bf=1)                                              │
│               /                                                      │
│             25(0)                                                     │
│   -> 平衡: 无失衡                                                     │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤5: 插入 40                                                        │
│   -> 树:     20(bf=-1)                                                │
│           /    \\                                                    │
│         10(0)  30(bf=-1)                                             │
│               /  \\                                                   │
│             25(0) 40(0)                                              │
│   -> 平衡: 无失衡                                                     │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤6: 插入 35                                                        │
│   -> 插入前: 节点30的平衡因子为-1，插入35后: 30(bf=-2) 失衡！            │
│   -> 失衡类型: RL型（右左）                                            │
│   -> 过程: 先对右子树(40)右旋，再对节点30左旋                           │
│   -> 旋转轴: 35                                                        │
│   -> 结果:     20(bf=0)                                               │
│           /    \\                                                    │
│         10(0)  35(bf=0)                                              │
│               /  \\                                                   │
│             30(0) 40(0)                                              │
│               \\    \\                                                │
│              25(0) 50(0)  <- 步骤7插入50后                             │
├─────────────────────────────────────────────────────────────────────┤
│ 步骤7: 插入 50                                                        │
│   -> 插入后: 节点35的平衡因子为-1，节点20的平衡因子为-1                 │
│   -> 平衡: 无失衡（树保持平衡）                                         │
│   -> 最终树:      35(bf=0)                                            │
│               /      \\                                              │
│             20(0)    40(0)                                           │
│            /   \\       \\                                            │
│          10(0) 30(0)   50(0)                                         │
│                  \\                                                   │
│                  25(0)                                                │
└─────────────────────────────────────────────────────────────────────┘
    """)

    # 验证最终树的平衡性
    print("\n" + "=" * 70)
    print("最终树平衡性验证")
    print("=" * 70)

    def check_balance(node: Optional[AVLNode]) -> Tuple[bool, int]:
        """检查树的平衡性，返回(是否平衡, 高度)"""
        if not node:
            return True, 0

        left_balanced, left_height = check_balance(node.left)
        right_balanced, right_height = check_balance(node.right)

        balanced = left_balanced and right_balanced and abs(left_height - right_height) <= 1
        height = 1 + max(left_height, right_height)

        return balanced, height

    is_balanced, _ = check_balance(avl.root)
    print(f"AVL树平衡性: {'✓ 完全平衡' if is_balanced else '✗ 不平衡'}")
    print(f"中序遍历结果: {avl.inorder_traversal()}")
    print(f"BST性质: {'✓ 保持' if avl.inorder_traversal() == sorted(avl.inorder_traversal()) else '✗ 破坏'}")

    print("\n" + "=" * 70)
    print("作业完成！")
    print("=" * 70)

AVL 树构建与旋转实操
插入序列: [30, 20, 10, 25, 40, 35, 50]

插入节点: 30
  -> 创建新节点 30
30(bf=0)


────────────────────────────────────────
第 1 步完成
当前树的中序遍历: [30]
────────────────────────────────────────

插入节点: 20
  -> 20 < 30，插入左子树
  -> 创建新节点 20
  -> 节点 30: 高度=2, 平衡因子=1
30(bf=1)
├── 20(bf=0)
└── None


────────────────────────────────────────
第 2 步完成
当前树的中序遍历: [20, 30]
────────────────────────────────────────

插入节点: 10
  -> 10 < 30，插入左子树
  -> 10 < 20，插入左子树
  -> 创建新节点 10
  -> 节点 20: 高度=2, 平衡因子=1
  -> 节点 30: 高度=3, 平衡因子=2
  ⚠️ 节点 30 发生 LL 型失衡！
  -> 执行右旋 (RR旋转)
  -> 旋转轴: 20
20(bf=0)
├── 10(bf=0)
└── 30(bf=0)


────────────────────────────────────────
第 3 步完成
当前树的中序遍历: [10, 20, 30]
────────────────────────────────────────

插入节点: 25
  -> 25 > 20，插入右子树
  -> 25 < 30，插入左子树
  -> 创建新节点 25
  -> 节点 30: 高度=2, 平衡因子=1
  -> 节点 20: 高度=3, 平衡因子=-1
20(bf=-1)
├── 10(bf=0)
└── 30(bf=1)
    ├── 25(bf=0)
    └── None


────────────────────────────────────────
第 4 步完成
当前树的中序遍历: [10, 20, 25, 30]
───────────────────────────────